<a href="https://colab.research.google.com/github/cursuri-inf/IPAP/blob/main/rampa/R3_probabilitati_entropie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caietul de rampă „Probabilități și entropie încrucișată” (rampa 3 din 3)

Un clasificator nu spune doar „Gentoo”: spune „Gentoo, cu probabilitatea 0,9”. Ca să-l antrenezi, ai nevoie de un număr care măsoară cât de greșite sunt aceste probabilități: entropia încrucișată, pierderea pe care ai văzut-o scăzând în caietul L2. Aici o construiești pas cu pas, din probabilități calculate pe pinguini, fără demonstrații.

**Pentru cine.** Caietul este pentru traseele Rampă și Nivelare Python, recomandate de diagnosticul de la cursul 1, și pornește de unde s-au oprit rampele 1 și 2. Îl parcurgi în săptămâna 3, înainte de laboratorul 3. Pe traseul Direct îl poți parcurge ca recapitulare.

**Cum lucrezi:**

1. Deschizi caietul cu butonul „Open in Colab” de mai sus. Colab avertizează că documentul nu a fost creat de Google: caietul vine din depozitul cursului, așa că alegi „Run anyway”.
2. Înainte să scrii ceva: File → Save a copy in Drive; lucrezi în copie.
3. Rulezi celulele în ordine, de sus în jos, cu Shift+Enter.
4. Unde vezi `None` sau `...`, completezi tu. Celula verifică singură răspunsul și scrie „corect” sau „nu încă”, cu un indiciu; nu îți arată rezolvarea.
5. La exercițiile „prezice”, scrii întâi ce crezi că va afișa codul, apoi îl rulezi și compari.
6. Notițele tale: la sfârșitul fiecărei secțiuni ai o celulă „Notițele mele”, în care scrii cu cuvintele tale ce ai înțeles și ce întrebări ai (dublu clic pe celulă ca s-o editezi). Poți lăsa și comentarii pe marginea din dreapta a oricărei celule: clic dreapta pe celulă, apoi „Add a comment” (sau Ctrl+Alt+M); comentariile rămân în copia ta din Drive.

Caietul nu se predă și nu se notează. La final, o celulă îți arată câte exerciții ai rezolvat; rezultatul îl discuți la tutoriat sau la punctul de control din săptămâna 5.

## Harta caietului

| Secțiunea | Ce înveți | Unde îți folosește |
|---|---|---|
| 1. Probabilități din frecvențe | cât de des apare o categorie în date | referința simplă, clasa cea mai frecventă |
| 2. Probabilități condiționate | probabilitatea, când știi deja ceva | ce poate prezice un model din trăsături |
| 3. Softmax | cum devin scorurile unui model probabilități | ieșirea oricărui clasificator |
| 4. Surpriza | `-log(p)`: cât de surprinzător e un rezultat | ideea din spatele pierderii |
| 5. Entropia încrucișată | pierderea unui clasificator, pe un exemplu și pe un lot | `nn.CrossEntropyLoss` din caietul L2 |
| 6. Două clase | sigmoida și entropia încrucișată binară | regresia logistică, la laboratorul 3 |
| 7. Legătura cu antrenarea | coborârea pe gradient micșorează pierderea | orice antrenare |

## 0. Pornire

Celula de mai jos încarcă bibliotecile, afișează versiunile și pregătește funcția `verifica`, care îți controlează răspunsurile. În Colab îi vezi doar titlul; codul se deschide cu dublu clic, dar nu trebuie citit. Funcția compară o amprentă a răspunsului tău cu amprenta rezolvării, deci nu conține rezolvările.

In [ ]:
# @title Pornire: rulează celula (Shift+Enter); codul ei nu trebuie citit
import hashlib, numbers, sys
import math, os
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

print("Python    ", sys.version.split()[0])
print("pandas    ", pd.__version__)
print("PyTorch   ", torch.__version__)
print("matplotlib", matplotlib.__version__)

REZULTATE = {}  # exercițiile rezolvate corect


def _normal(v):
    if hasattr(v, "tolist"):
        v = v.tolist()  # tensorii devin numere sau liste
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, numbers.Number):
        return f"{float(v):.2f}"  # numerele se compară cu 2 zecimale
    if isinstance(v, (list, tuple)):
        return ",".join(_normal(x) for x in v)
    return str(v).strip().lower()


def _necompletat(v):
    if isinstance(v, (list, tuple)):
        return any(_necompletat(x) for x in v)
    return v is None or v is ...


def verifica(exercitiu, raspuns, amprenta, indiciu):
    """Scrie „corect” sau „nu încă”, fără să arate rezolvarea."""
    if _necompletat(raspuns):
        print(f"{exercitiu}: încă necompletat. Înlocuiește None sau ... și rulează din nou.")
    elif hashlib.sha256(_normal(raspuns).encode("utf-8")).hexdigest()[:12] == amprenta:
        REZULTATE[exercitiu] = True
        print(f"{exercitiu}: corect.")
    else:
        REZULTATE[exercitiu] = False
        print(f"{exercitiu}: nu încă. Indiciu: {indiciu}")

În acest caiet, verificarea compară numerele cu **două zecimale** (0,44 și 0,4419 sunt același răspuns). „log” înseamnă logaritmul natural (ln), ca în PyTorch: `math.log` și `torch.log`.

## 1. Probabilități din frecvențe

Cea mai simplă probabilitate: de câte ori apare o categorie, împărțit la numărul total de cazuri. Din tabelul cu pinguini, probabilitatea ca un pinguin ales la întâmplare să fie Gentoo este numărul pinguinilor Gentoo împărțit la numărul tuturor pinguinilor.

Celula citește tabelul de la cursul 2; dacă adresa nu răspunde, încarci `penguins.csv` din Campus Virtual în Colab (panoul Files, butonul Upload) și rulezi din nou.

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
pinguini = pd.read_csv("penguins.csv") if os.path.exists("penguins.csv") else pd.read_csv(URL)

numar = pinguini["species"].value_counts()
total = len(pinguini)
print(numar, "\n")
print("Total:", total)
print("P(Gentoo) =", numar["Gentoo"], "/", total, "=", round(numar["Gentoo"] / total, 4))

### Exercițiul 1 (completează)

Calculează probabilitatea ca un pinguin ales la întâmplare să fie Adelie, folosind `numar` și `total`.

In [ ]:
p_adelie = None   # înlocuiește None

print("P(Adelie) =", p_adelie)
verifica("Exercițiul 1", p_adelie, "26ccbceee7ef", "numar[\"Adelie\"] împărțit la total")

### Exercițiul 2 (prezice)

Cât crezi că dă suma probabilităților celor trei specii? De ce? Scrie predicția, apoi rulează.

Răspunsul tău: …

In [ ]:
probabilitati = numar / total
print(probabilitati.round(4))
print("Suma:", probabilitati.sum())

Referința simplă de la cursul 2 vine de aici: un „model” care răspunde mereu cu specia cea mai frecventă are dreptate în proporția acelei specii. Orice clasificator trebuie să o depășească clar.

### Notițele mele

…

## 2. Probabilități condiționate

Probabilitatea condiționată răspunde la întrebarea: cât de probabil e ceva, când știu deja altceva? De exemplu, P(Gentoo | Biscoe) se citește „probabilitatea ca un pinguin să fie Gentoo, știind că a fost măsurat pe insula Biscoe”. Se calculează la fel, dar numai pe rândurile care îndeplinesc condiția.

Tabelul încrucișat (`pd.crosstab`) numără pinguinii pe insulă și pe specie:

In [ ]:
tabel = pd.crosstab(pinguini["island"], pinguini["species"], margins=True, margins_name="Total")
tabel

### Exercițiul 3 (completează)

Calculează P(Gentoo | Biscoe) din tabel: câți pinguini Gentoo sunt pe Biscoe, împărțit la câți pinguini sunt pe Biscoe în total. Un element al tabelului se citește cu `tabel.loc["rând", "coloană"]`.

In [ ]:
p_gentoo_biscoe = None   # înlocuiește None

print("P(Gentoo | Biscoe) =", p_gentoo_biscoe)
verifica("Exercițiul 3", p_gentoo_biscoe, "5ecbc825ba5c", "tabel.loc[\"Biscoe\", \"Gentoo\"] împărțit la tabel.loc[\"Biscoe\", \"Total\"]")

### Exercițiul 4 (prezice)

Uită-te la tabel: cât este P(Adelie | Torgersen)? Ce ar putea spune un model care știe doar insula, pentru un pinguin de pe Torgersen? Dar pentru unul de pe Dream? Scrie predicția, apoi rulează.

Răspunsul tău: …

In [ ]:
pe_insula = pd.crosstab(pinguini["island"], pinguini["species"], normalize="index")   # fiecare rând însumează 1
pe_insula.round(2)

Fiecare rând al tabelului de mai sus este o **distribuție de probabilitate** pe cele trei specii: numere între 0 și 1, cu suma 1. Exact asta produce un clasificator pentru fiecare exemplu.

### Notițele mele

…

## 3. Softmax: de la scoruri la probabilități

Ultimul strat al unei rețele dă câte un **scor** pentru fiecare clasă: numere oarecare, pozitive sau negative. Softmax le transformă în probabilități: ridică e la puterea fiecărui scor (toate devin pozitive), apoi împarte la sumă (suma devine 1). Scorul cel mai mare primește probabilitatea cea mai mare. Așa a obținut, la cursul 1, modelul ResNet probabilitățile celor 1.000 de categorii (`.softmax(dim=1)`).

In [ ]:
scoruri = torch.tensor([2.0, 0.5, -1.0])   # scorurile pentru Adelie, Chinstrap, Gentoo
exp = torch.exp(scoruri)
print("e la puterea scorurilor:", exp)
print("Suma lor:", exp.sum())
print("Softmax, pas cu pas:", exp / exp.sum())
print("Softmax din PyTorch: ", torch.softmax(scoruri, dim=0))

### Exercițiul 5 (completează)

Ce probabilitate dă softmax primei clase (Adelie)? Răspunsul este un singur număr: elementul cu indicele 0 al vectorului de probabilități.

In [ ]:
p_prima = None   # înlocuiește None

print("Probabilitatea clasei Adelie:", p_prima)
verifica("Exercițiul 5", p_prima, "8761fef59928", "torch.softmax(scoruri, dim=0)[0]")

### Exercițiul 6 (prezice)

Ce se întâmplă cu probabilitățile dacă aduni 10 la toate scorurile? Dar dacă înmulțești toate scorurile cu 3? Scrie predicția, apoi rulează.

Răspunsul tău: …

In [ ]:
print(torch.softmax(scoruri, dim=0))
print(torch.softmax(scoruri + 10, dim=0))
print(torch.softmax(scoruri * 3, dim=0))

### Notițele mele

…

## 4. Surpriza: `-log(p)`

Cât de surprins ești când se întâmplă ceva care avea probabilitatea p? Dacă era sigur (p = 1), deloc; dacă era foarte puțin probabil, foarte tare. Măsura folosită este `-log(p)`: 0 pentru p = 1 și din ce în ce mai mare când p se apropie de 0.

In [ ]:
p = torch.linspace(0.01, 1, 200)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(p, -torch.log(p), color="#161616")
for pi in (0.9, 0.5, 0.1):
    ax.scatter([pi], [-math.log(pi)], color="#9B1B30", zorder=3)
    ax.annotate(f"p = {pi}: {-math.log(pi):.2f}", (pi, -math.log(pi)), textcoords="offset points", xytext=(8, 6))
ax.set_xlabel("probabilitatea p")
ax.set_ylabel("surpriza, -log(p)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Notițele mele

…

## 5. Entropia încrucișată: pierderea unui clasificator

Ideea pierderii: te uiți ce probabilitate a dat modelul **clasei corecte** și măsori surpriza. Dacă modelul a dat 0,9 clasei corecte, pierderea e mică (0,11); dacă a dat 0,1, pierderea e mare (2,30). Pierderea penalizează mai ales greșelile făcute cu siguranță. Aceasta este **entropia încrucișată** (cross-entropy) pentru un exemplu: `-log(probabilitatea clasei corecte)`.

### Exercițiul 7 (completează)

Modelul a dat probabilitățile 0,7, 0,2 și 0,1 pentru Adelie, Chinstrap și Gentoo, iar pinguinul este Adelie. Calculează pierderea.

In [ ]:
probabilitati_model = torch.tensor([0.7, 0.2, 0.1])
clasa_corecta = 0   # Adelie

pierdere = None   # înlocuiește None

print("Pierderea:", pierdere)
verifica("Exercițiul 7", pierdere, "9c9e5b4a9207", "minus logaritmul probabilității clasei corecte: -torch.log(probabilitati_model[clasa_corecta])")

Pe un lot de exemple, pierderea este **media** pierderilor exemplelor. Asta afișa caietul L2 la fiecare epocă.

### Exercițiul 8 (completează)

Pe trei pinguini, modelul a dat clasei corecte probabilitățile 0,7, 0,4 și 0,9. Calculează pierderea medie pe lot.

In [ ]:
p_corecte = torch.tensor([0.7, 0.4, 0.9])

pierdere_lot = None   # înlocuiește None

print("Pierderea medie pe lot:", pierdere_lot)
verifica("Exercițiul 8", pierdere_lot, "ad9dadb817c4", "-torch.log(p_corecte) dă pierderea fiecărui exemplu; apoi .mean()")

### Exercițiul 9 (interpretează)

Două modele clasifică aceiași trei pinguini și ambele au acuratețea 100% (clasa corectă primește de fiecare dată probabilitatea cea mai mare). Modelul A dă clasei corecte 0,55 de fiecare dată, modelul B dă 0,95. Ce pierdere are fiecare? De ce antrenăm modelele pe pierdere, nu direct pe acuratețe?

In [ ]:
for nume, p in (("A", 0.55), ("B", 0.95)):
    print(f"Modelul {nume}: pierderea", round(-math.log(p), 3))

Răspunsul tău: …

**În PyTorch**, pierderea se calculează cu `F.cross_entropy` (sau `nn.CrossEntropyLoss`, ca în caietul L2). Atenție: funcția primește **scorurile** (înainte de softmax), nu probabilitățile; aplică singură softmax și logaritmul. Țintele sunt indicii claselor corecte.

### Exercițiul 10 (completează)

Pentru două exemple, scorurile modelului și clasele corecte sunt date mai jos. Calculează pierderea medie cu `F.cross_entropy`. După ce completezi, celula afișează și calculul pas cu pas, ca să compari.

In [ ]:
scoruri_lot = torch.tensor([[2.0, 0.5, -1.0],    # exemplul 1
                            [0.0, 1.0, 0.0]])    # exemplul 2
tinte = torch.tensor([0, 2])                     # clasa corectă: Adelie, apoi Gentoo

pierdere_pytorch = None   # înlocuiește None

pas_cu_pas = -torch.log(torch.softmax(scoruri_lot, dim=1)[[0, 1], tinte]).mean()
print("F.cross_entropy:", pierdere_pytorch)
if pierdere_pytorch is not None:
    print("Calculul pas cu pas, pentru comparație:", pas_cu_pas)
verifica("Exercițiul 10", pierdere_pytorch, "26210f2c9712", "F.cross_entropy(scoruri_lot, tinte): întâi scorurile, apoi țintele")

### Notițele mele

…

## 6. Două clase: sigmoida și entropia încrucișată binară

Când sunt doar două clase (da sau nu, Gentoo sau nu), modelul dă un singur scor, iar **sigmoida** îl transformă în probabilitatea clasei „da”: `1 / (1 + e^(-scor))`. Scorul 0 dă 0,5; scorurile mari dau probabilități aproape de 1, cele foarte negative, aproape de 0. Regresia logistică, de la laboratorul 3, este exact asta: un produs scalar (rampa 2), trecut prin sigmoidă.

In [ ]:
s = torch.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(s, torch.sigmoid(s), color="#161616")
ax.axhline(0.5, color="#BDBDBD", linewidth=1)
ax.set_xlabel("scorul")
ax.set_ylabel("probabilitatea clasei „da”")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
print("sigmoid(0) =", torch.sigmoid(torch.tensor(0.0)).item())

Pierderea are aceeași idee: surpriza față de răspunsul corect. Dacă răspunsul corect este „da” (y = 1), pierderea este `-log(p)`; dacă este „nu” (y = 0), pierderea este `-log(1 - p)`. În PyTorch: `F.binary_cross_entropy(p, y)` pentru probabilități.

### Exercițiul 11 (completează)

Modelul dă probabilitatea 0,8 că pinguinul este Gentoo, iar pinguinul chiar este Gentoo (y = 1). Calculează pierderea.

In [ ]:
p_gentoo = torch.tensor(0.8)
y = torch.tensor(1.0)

pierdere_binara = None   # înlocuiește None

print("Pierderea:", pierdere_binara)
if pierdere_binara is not None:
    print("Cu F.binary_cross_entropy, pentru comparație:", F.binary_cross_entropy(p_gentoo, y))
verifica("Exercițiul 11", pierdere_binara, "dc08a4e8bbdd", "răspunsul corect este „da”, deci pierderea este -torch.log(p_gentoo)")

### Notițele mele

…

## 7. Legătura cu antrenarea

Rampa 2 a arătat coborârea pe gradient; aici o aplici chiar pe entropia încrucișată. Pornești de la scoruri egale (toate clasele au 1/3) pentru un pinguin Adelie și faci pași în sens opus gradientului pierderii. Pierderea scade, iar probabilitatea clasei corecte crește: așa învață ultimul strat al unei rețele.

### Exercițiul 12 (modifică și interpretează)

Rulează celula, apoi schimbă rata de învățare `RATA` (0.1, 1.0, 5.0) și numărul de pași. Ce se întâmplă cu pierderea și cu probabilitatea clasei corecte? Poate pierderea să ajungă exact la 0?

In [ ]:
RATA = 0.5
PASI = 30

scor = torch.zeros(3, requires_grad=True)   # scorurile pentru Adelie, Chinstrap, Gentoo
tinta = torch.tensor([0])                     # pinguinul este Adelie
istoric = []
for pas in range(PASI):
    pierdere = F.cross_entropy(scor.unsqueeze(0), tinta)
    istoric.append(pierdere.item())
    pierdere.backward()                       # gradientul pierderii față de scoruri
    with torch.no_grad():
        scor -= RATA * scor.grad              # pasul de coborâre
        scor.grad.zero_()

print("Probabilitățile la final:", torch.softmax(scor, dim=0).detach().round(decimals=3))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(istoric, marker="o", markersize=3, color="#9B1B30")
ax.set_xlabel("pasul")
ax.set_ylabel("pierderea")
ax.set_title(f"Entropia încrucișată în timpul coborârii (rata {RATA})")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Răspunsul tău: …

### Notițele mele

…

## Verificarea finală

Rulează Runtime → Restart session and run all, apoi uită-te la celula de mai jos: arată ce exerciții sunt rezolvate și pe care le mai ai de revăzut.

In [ ]:
EXERCITII = ["Exercițiul 1", "Exercițiul 3", "Exercițiul 5", "Exercițiul 7", "Exercițiul 8", "Exercițiul 10", "Exercițiul 11"]
corecte = [e for e in EXERCITII if REZULTATE.get(e)]
print(f"Exerciții verificate automat, rezolvate: {len(corecte)} din {len(EXERCITII)}")
for e in EXERCITII:
    print(" ", e, "rezolvat" if REZULTATE.get(e) else "de revăzut")

## Ce ai învățat

- O probabilitate din date este o frecvență relativă; probabilitățile tuturor categoriilor însumează 1.
- O probabilitate condiționată se calculează numai pe cazurile care îndeplinesc condiția, de exemplu pe o singură insulă.
- Softmax transformă scorurile unui model în probabilități; contează doar diferențele dintre scoruri.
- Surpriza `-log(p)` este 0 pentru un rezultat sigur și crește mult când p se apropie de 0.
- Entropia încrucișată este surpriza față de clasa corectă, în medie pe lot; `F.cross_entropy` primește scorurile, nu probabilitățile.
- Cu două clase, sigmoida dă probabilitatea clasei „da”, iar pierderea este entropia încrucișată binară.
- Antrenarea înseamnă coborâre pe gradient pe această pierdere.

## Dacă te blochezi

- Citește din nou mesajul de eroare de la sfârșit: spune pe ce rând și de ce s-a oprit codul.
- Întreabă-ți colegul de pereche sau vino la tutoriatul online.
- Folosește un asistent AI ca tutor, cu rețeta din cursul 1: „Explică-mi pas cu pas ce face linia aceasta, fără să-mi dai rezolvarea exercițiului.”

## Mai departe

La laboratorul 3 pui totul cap la cap: regresia liniară (produs scalar și eroare pătratică) și regresia logistică (produs scalar, sigmoidă și entropie încrucișată binară), antrenate prin coborâre pe gradient, cu gradientul calculat de PyTorch.